# 🧠 Brain-Master — Worker de Inferencia en Colab (GPU T4 gratis)

Levanta el worker gRPC completo en la T4 de Colab y lo expone a internet:

- **gRPC (:50051) → túnel `bore`** (TCP puro, ideal para gRPC/HTTP2, sin registro)
- **Artefactos HTTP (:50052) → túnel `cloudflared`** (descarga de PNG/MP4)

La celda 4 imprime las dos variables de configuración del gateway **y además
las publica sola en un topic efímero de ntfy.sh**: en tu máquina basta correr
`connect-gateway.ps1 -NtfyTopic <topic>` y el gateway se configura solo —
cero copy/paste.

⚠️ La sesión free dura horas y al cerrar la pestaña se corta: re-ejecuta la
celda 4 para reconectar (los túneles cambian de puerto/URL).

## 1 — Verificar GPU

In [ ]:
# Runtime > Change runtime type > T4 GPU (imprescindible antes de ejecutar)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2 — Obtener el código

In [ ]:
import os
# Opción A: clonar el repo público (ya contiene inference-worker/).
#           Si el repo llegara a ser privado: sube inference-worker.zip
#           (opción B) o clona con token: https://<TU_TOKEN>@github.com/synappta4ai/brain-master.git
REPO_URL = "https://github.com/synappta4ai/brain-master.git"
# Opción B: subir la carpeta inference-worker.zip con el panel de archivos
#           y descomprimirla en /content/inference-worker

if os.path.exists("/content/inference-worker/server.py"):
    print("usando /content/inference-worker (subido a mano)")
else:
    if os.path.isdir("/content/brain-master/.git"):
        # El disco de Colab sobrevive a 'Reiniciar entorno': sin este pull,
        # el clon queda con codigo viejo para siempre.
        !git -C /content/brain-master pull --ff-only
    else:
        !git clone --depth 1 $REPO_URL /content/brain-master
    !ln -sfn /content/brain-master/inference-worker /content/inference-worker
%cd /content/inference-worker


## 3 — Dependencias y túneles (bore + cloudflared)
> ℹ️ **El WARNING de pip al final es esperado**: el resolver avisa que
> `protobuf 7.x` (nuestro stack) pisa versiones de paquetes preinstalados de
> Colab (`ydf`, `google-ai-generativelanguage`, `grpcio-status` — SDKs de
> Gemini que el worker no usa). Todo lo que el worker necesita se instala bien
> y así se generó imágenes reales en la T4. Seguí a la celda 4 sin miedo.

In [ ]:
# Stack de IA (Colab ya trae torch+CUDA)
!pip install -q grpcio==1.84.0 grpcio-tools==1.84.0 "protobuf>=5.29.3" \
    "diffusers>=0.36.0" "transformers>=4.57.0" tokenizers accelerate safetensors \
    sentencepiece einops open_clip_torch imageio imageio-ffmpeg Pillow psutil bitsandbytes

In [ ]:
# bore v0.6.0 (túnel TCP para gRPC) + cloudflared (túnel HTTP para artefactos)
!curl -sL -o /tmp/bore.tar.gz https://github.com/ekzhang/bore/releases/download/v0.6.0/bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz
!tar -xzf /tmp/bore.tar.gz -C /usr/local/bin && chmod +x /usr/local/bin/bore
!curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!bore --version && cloudflared --version

## 4 — Arrancar worker + túneles ⭐ (re-ejecutar para reconectar)

In [ ]:
import subprocess, time, re, os

os.environ["BM_OUTPUT_DIR"] = "/content/outputs"
os.makedirs("/content/outputs", exist_ok=True)

# Mata instancias previas (re-conexión)
!pkill -f '[p]ython server.py' 2>/dev/null; pkill -f '[b]ore local' 2>/dev/null; pkill -f '[c]loudflared' 2>/dev/null   # [x] trick: evita que pkill se mate a si mismo
time.sleep(2)

worker = subprocess.Popen(["python", "server.py"],
                          stdout=open("/content/worker.log", "a"),
                          stderr=subprocess.STDOUT)
time.sleep(6)

def wait_for(path, pattern, tries=30):
    """Espera el patrón y devuelve el match (el caller elige group 0 o 1)."""
    for _ in range(tries):
        time.sleep(1)
        m = re.search(pattern, open(path).read())
        if m:
            return m
    raise RuntimeError(f"patrón no apareció en {path}:\n" + open(path).read())

# --- gRPC por bore (TCP puro). SIN --port fijo: bore.pub asigna uno libre
#     (pedir 50051 fijo colisiona con otros usuarios del relay público).
bore = subprocess.Popen(["bore", "local", "50051", "--to", "bore.pub"],
                        stdout=open("/content/bore.log", "w"), stderr=subprocess.STDOUT)
bore_port = wait_for("/content/bore.log", r"listening at bore\.pub:(\d+)", 30).group(1)

# --- Artefactos por cloudflared (HTTP)
cf = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:50052",
                       "--no-autoupdate"],
                      stdout=open("/content/cf.log", "w"), stderr=subprocess.STDOUT)
art_url = wait_for("/content/cf.log", r"https://[a-z0-9-]+\.trycloudflare\.com").group(0)

print("\n" + "=" * 64)
print("PYTHON_WORKER_HOST=bore.pub:" + bore_port)
print("BM_WORKER_ARTIFACT_BASE=" + art_url)
print("=" * 64)
print("Auto-announce activo: copia el ntfy.topic y usa connect-gateway.ps1 -NtfyTopic")


# --- Auto-announce ntfy.sh (cero copy/paste) ------------------------------
# Publica las dos variables en un topic efímero de ntfy.sh. En tu máquina:
#   .\deploy\colab\connect-gateway.ps1 -NtfyTopic "<este topic>"
# El gateway se configura solo. El topic es aleatorio y efímero (ntfy retiene horas
# por defecto); no viajan secretos, solo los túneles de esta sesión.
import json as _json, secrets, requests

NTFY = "https://ntfy.sh"
topic = "bm-brain-master-tunnels-v1"   # FIJO: lo consume watch-announce.py (deploy/colab/)

payload = {
    "PYTHON_WORKER_HOST": "bore.pub:" + bore_port,
    "BM_WORKER_ARTIFACT_BASE": art_url,
}
resp = requests.post(
    f"{NTFY}/{topic}",
    data=_json.dumps(payload).encode(),
    headers={"Title": "brain-master worker UP", "Priority": "high", "Tags": "white_check_mark"},
    timeout=15,
)
resp.raise_for_status()

print()
print("ntfy.topic = " + topic)

## 5 — Smoke test local: difusión real en la T4

In [ ]:
# Primera generación REAL (descarga SD-Tiny-Test ~100MB la primera vez).
# Corre directo en el kernel de Colab: los heredocs (!python - <<EOF) no
# funcionan en el notebook y asyncio.run() choca con el event loop del
# kernel → se usa await a nivel de celda.
import sys
sys.path.insert(0, '.')
import grpc
import inference_pb2, inference_pb2_grpc

async def main():
    async with grpc.aio.insecure_channel('127.0.0.1:50051') as ch:
        stub = inference_pb2_grpc.InferenceServiceStub(ch)
        cat = await stub.ListModels(inference_pb2.ListModelsRequest())
        print(f"catálogo: {len(cat.models)} modelos | cuda={cat.cuda_available} ({cat.device_name})")
        req = inference_pb2.MediaGenerationRequest(
            job_id='colab-smoke', mode='image', model_name='SD-Tiny-Test',
            prompt='a neon city at dusk, cinematic', steps=4, width=512, height=512)
        async for ev in stub.GenerateMedia(req):
            print(f"  {ev.status:14s} {ev.percentage:5.1f}%")

await main()


## 6 — Verificar los túneles desde fuera (como lo hará el gateway)

In [ ]:
# Verifica ambos túneles desde fuera, tal como lo hará el gateway.
import re, sys
sys.path.insert(0, '.')
import grpc
import inference_pb2, inference_pb2_grpc

# 1) gRPC a través de bore.pub (la ruta exacta que usará el gateway)
port = re.search(r'listening at bore\.pub:(\d+)', open('/content/bore.log').read()).group(1)

async def check_grpc():
    async with grpc.aio.insecure_channel(f'bore.pub:{port}') as ch:
        stub = inference_pb2_grpc.InferenceServiceStub(ch)
        tel = await stub.GetGpuTelemetry(inference_pb2.GpuTelemetryRequest())
        print(f"✓ gRPC por bore.pub:{port} → device={tel.device_name}, VRAM libre={tel.free_vram_mb}MB")

await check_grpc()

# 2) Artefacto vía cloudflared (la URL que usará el front)
import requests
url = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read()).group(0)
r = requests.get(url + '/healthz', timeout=15)
print('✓ artifact server por túnel:', r.status_code, r.json())


## 7 — Conectar tu gateway (en tu máquina)

**Modo auto-announce (recomendado, cero copy/paste):** copia de la salida de
la celda 4 solo la línea `ntfy.topic = ...` y ejecuta:

```powershell
.\deploy\colab\connect-gateway.ps1 -NtfyTopic "<topic>"
```

El script consume el topic, reinicia el gateway con las variables correctas y
verifica health + catálogo + job de prueba.

**Modo manual** (si ntfy.sh no estuviera disponible), con las dos variables
impresas:

```powershell
.\deploy\colab\connect-gateway.ps1 -WorkerHost "bore.pub:PORT" -ArtifactBase "https://xxx.trycloudflare.com"
```

Linux/macOS:

```bash
export PYTHON_WORKER_HOST=bore.pub:PORT BM_WORKER_ARTIFACT_BASE=https://xxx.trycloudflare.com
# reinicia el gateway con esas variables
```

In [ ]:
# Monitor del worker (detener con ■)
!tail -f /content/worker.log